# 00_build_fusion_input.ipynb
## Purpose
Build fusion input by left-joining RB/UNSUP outputs onto the master segment universe (segments_index).

## Expected inputs
- `data/segments_index.csv`
- `outputs/rb_output.csv`
- `outputs/unsup_output.csv`

## Expected outputs
- `data/dataset_merged_with_unsup_rb_aspects_ALLCOLS.csv`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

# Build fusion input (UNSUP + RB merge)
This notebook constructs `data/dataset_merged_with_unsup_rb_aspects_ALLCOLS.csv` by merging the UNSUP and RB branch outputs on `seg_key = comment_id__seg_id`.

In [ ]:
import pandas as pd
from pathlib import Path

# Resolve repository root robustly (works whether run from repo root or from notebooks/)
CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not locate repo root containing the 'data/' folder. Run from repo root or from 'notebooks/'.")

SEGMENTS_PATH = REPO_ROOT / "data" / "segments_index.csv"
UNSUP_PATH = REPO_ROOT / "outputs" / "unsup_output.csv"
RB_PATH    = REPO_ROOT / "outputs" / "rb_output.csv"
OUT_PATH   = REPO_ROOT / "data" / "dataset_merged_with_unsup_rb_aspects_ALLCOLS.csv"

segments = pd.read_csv(SEGMENTS_PATH)
unsup = pd.read_csv(UNSUP_PATH)
rb    = pd.read_csv(RB_PATH)

# Ensure unique key
for df in (segments, unsup, rb):
    df["comment_id"] = df["comment_id"].astype(str)
    df["seg_id"] = df["seg_id"].astype(str)
    if "seg_key" not in df.columns:
        df["seg_key"] = df["comment_id"] + "__" + df["seg_id"]

# Core columns to attach from each branch
unsup_keep = [c for c in ["seg_key","aspect_terms_unsup","top_aspect_terms_unsup","top_aspect_cos_unsup",
                         "terms_ranked","cos_ranked","fallback_mode","cluster_unsup"] if c in unsup.columns]
rb_keep = [c for c in ["seg_key","aspect_terms_rb_clean","aspect_terms_rb","aspect_terms_rb_raw",
                      "rb_lang_used","seg_text_processed","seg_text_clean","interj_removed"] if c in rb.columns]

unsup_core = unsup[unsup_keep].copy()
rb_core = rb[rb_keep].copy()

# Start from the master segment universe (fairness protocol)
merged = segments.merge(rb_core, on="seg_key", how="left").merge(unsup_core, on="seg_key", how="left")

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUT_PATH, index=False)
print("Wrote:", OUT_PATH, "rows:", len(merged), "unique seg_key:", merged["seg_key"].nunique())